# Fingerflex 数据可视化

本 notebook 用于快速浏览 ECoG 电极信号、手指轨迹和 cue 标签。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# 修改为你的本地数据路径
MAT_PATH = Path('/path/to/01_fingerflex.mat')

mat = loadmat(MAT_PATH)
data = np.asarray(mat['data'], dtype=np.float32)  # (time, channels)
flex = np.asarray(mat['flex'], dtype=np.float32)  # (time, 5)
cue = np.asarray(mat['cue']).squeeze().astype(int)

print('data:', data.shape)
print('flex:', flex.shape)
print('cue:', cue.shape)

In [ ]:
# 1) 随机若干电极的时域波形
rng = np.random.default_rng(0)
chs = rng.choice(data.shape[1], size=min(8, data.shape[1]), replace=False)

start, end = 0, min(10_000, data.shape[0])  # 前10秒
t = np.arange(start, end) / 1000.0

fig, axes = plt.subplots(len(chs), 1, figsize=(12, 1.8 * len(chs)), sharex=True)
if len(chs) == 1:
    axes = [axes]

for ax, ch in zip(axes, chs):
    ax.plot(t, data[start:end, ch], lw=0.8)
    ax.set_ylabel(f'ch{ch}')
axes[-1].set_xlabel('Time (s)')
fig.suptitle('Random channel traces')
plt.tight_layout()

In [ ]:
# 2) 五根手指 flex 轨迹
finger_names = ['thumb', 'index', 'middle', 'ring', 'little']
start, end = 0, min(20_000, flex.shape[0])  # 前20秒
t = np.arange(start, end) / 1000.0

plt.figure(figsize=(12, 4))
for i, name in enumerate(finger_names):
    plt.plot(t, flex[start:end, i], label=name, lw=1.2)
plt.title('Finger flex trajectories')
plt.xlabel('Time (s)')
plt.ylabel('Flexion value')
plt.legend(ncol=5)
plt.tight_layout()

In [ ]:
# 3) cue 标签随时间变化
start, end = 0, min(20_000, cue.shape[0])
t = np.arange(start, end) / 1000.0

plt.figure(figsize=(12, 2.5))
plt.step(t, cue[start:end], where='post')
plt.yticks(range(6), ['rest','thumb','index','middle','ring','little'])
plt.title('Cue timeline')
plt.xlabel('Time (s)')
plt.tight_layout()

In [ ]:
# 4) 手指间相关性热图
corr = np.corrcoef(flex.T)

plt.figure(figsize=(5, 4))
im = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
plt.xticks(range(5), finger_names, rotation=30)
plt.yticks(range(5), finger_names)
plt.title('Correlation between finger trajectories')
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout()

In [ ]:
# 5) 单电极功率谱（可用于挑选噪声通道）
from scipy.signal import welch

channel = 0
fs = 1000
f, pxx = welch(data[:, channel], fs=fs, nperseg=2048)

plt.figure(figsize=(7, 4))
plt.semilogy(f, pxx)
plt.xlim(0, 200)
plt.xlabel('Frequency (Hz)')
plt.ylabel('PSD')
plt.title(f'Power spectral density - channel {channel}')
plt.tight_layout()

> 后续可扩展：
> - 按 cue 分段后比较不同手指任务的通道响应均值。
> - 可视化每个电极与每个手指的相关矩阵。
> - 增加时频图（spectrogram）和高伽马包络。